# 📊 Sunum Grafikleri Üretici — Slide 2, 4, 5, 7, 8, 9

| Dosya | Slide |
|-------|-------|
| `slide2_channel_comparison_constellation.png` | 2 |
| `slide4_sample_iq_waveforms.png` | 4 |
| `slide5_fading_heatmap.png` | 5 |
| `slide7_training_strategy_mcldnn.png` | 7 (sol) |
| `slide7_training_strategy_petcgdnn.png` | 7 (sağ) |
| `slide8_model_comp_f1_rayleigh.png` | 8 (sol) |
| `slide8_model_comp_mcc_rayleigh.png` | 8 (sağ) |
| `slide9_confusion_matrix_comparison.png` | 9 |

In [ ]:
# 1. Ortam Kurulumu
from google.colab import drive
drive.mount('/content/drive')

import os, sys, pickle
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'figure.facecolor': 'white'})

PROJECT_DIR = '/content/drive/MyDrive/AMR-Project'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
FT_DIR = os.path.join(PROJECT_DIR, 'fine_tuning_results')
SAVE_DIR = os.path.join(PROJECT_DIR, 'Sunum_Grafikleri')
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Çıktı: {SAVE_DIR}')

## SLIDE 2, 4, 5 — Dataset Görselleri

In [ ]:
# Dataset'leri yükle
DS = {'AWGN': 'RML2016.10a_dict.pkl', 'Rayleigh': 'RML2016.10a_rayleigh.pkl',
      'Rician K=3': 'RML2016.10a_rician_K3.pkl', 'Rician K=10': 'RML2016.10a_rician_K10.pkl'}
datasets = {}
for label, f in DS.items():
    p = os.path.join(PROJECT_DIR, f)
    if os.path.exists(p):
        datasets[label] = pickle.load(open(p, 'rb'), encoding='iso-8859-1')
        print(f'  ✅ {label}')
    else:
        print(f'  ❌ {label}: {p}')
ref = list(datasets.values())[0]
mods_all = sorted(set(k[0] for k in ref.keys()))
snrs_all = sorted(set(k[1] for k in ref.keys()))

In [ ]:
# SLIDE 2 — Constellation Karşılaştırması
ch_colors = ['#2196F3', '#F44336', '#FF9800', '#4CAF50']
fig, axes = plt.subplots(1, 4, figsize=(18, 4), dpi=150)
for col, (ch, d) in enumerate(datasets.items()):
    X = d[('QPSK', 10)]
    axes[col].scatter(X[:40,0,:].flatten(), X[:40,1,:].flatten(), s=0.5, alpha=0.15, color=ch_colors[col])
    axes[col].set_aspect('equal'); axes[col].set_title(ch, fontsize=12, fontweight='bold')
    axes[col].grid(True, alpha=0.2); axes[col].tick_params(labelsize=8)
fig.suptitle('QPSK Constellation — SNR=10 dB', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'slide2_channel_comparison_constellation.png'), dpi=300, bbox_inches='tight')
plt.show()
print('✅ slide2_channel_comparison_constellation.png')

In [ ]:
# SLIDE 4 — IQ Dalga Formları
vis_mods = ['BPSK', 'QPSK', '8PSK', 'QAM16', 'QAM64', 'WBFM']
fig, axes = plt.subplots(2, 3, figsize=(15, 6), dpi=150)
for idx, mod in enumerate(vis_mods):
    r, c = idx // 3, idx % 3
    X = datasets['AWGN'][(mod, 10)]
    axes[r,c].plot(X[0,0,:], label='I', color='#1D3557', lw=0.9)
    axes[r,c].plot(X[0,1,:], label='Q', color='#E63946', lw=0.9, alpha=0.8)
    axes[r,c].set_title(mod, fontsize=11, fontweight='bold')
    axes[r,c].grid(True, alpha=0.2); axes[r,c].legend(fontsize=8, loc='upper right')
fig.suptitle('Sample IQ Waveforms — SNR=10 dB (AWGN)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'slide4_sample_iq_waveforms.png'), dpi=300, bbox_inches='tight')
plt.show()
print('✅ slide4_sample_iq_waveforms.png')

In [ ]:
# SLIDE 5 — Fading Heatmap
fading_chs = {k: v for k, v in datasets.items() if k != 'AWGN'}
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)
for idx, (ch, d) in enumerate(fading_chs.items()):
    ratio = np.zeros((len(mods_all), len(snrs_all)))
    for i, mod in enumerate(mods_all):
        for j, snr in enumerate(snrs_all):
            pa = np.mean(datasets['AWGN'][(mod,snr)]**2)
            pf = np.mean(d[(mod,snr)]**2)
            ratio[i,j] = pf/pa if pa > 0 else 1.0
    im = axes[idx].imshow(ratio, aspect='auto', cmap='RdYlGn', vmin=0.3, vmax=1.7)
    axes[idx].set_title(ch, fontsize=12, fontweight='bold')
    axes[idx].set_xticks(range(0, len(snrs_all), 2))
    axes[idx].set_xticklabels([str(s) for s in snrs_all[::2]], fontsize=8)
    axes[idx].set_xlabel('SNR (dB)')
    axes[idx].set_yticks(range(len(mods_all)))
    axes[idx].set_yticklabels(mods_all, fontsize=8)
    if idx == 0: axes[idx].set_ylabel('Modulation')
    plt.colorbar(im, ax=axes[idx], shrink=0.8)
fig.suptitle('Channel Distortion Heatmap — Power Ratio (Faded / AWGN)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'slide5_fading_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()
print('✅ slide5_fading_heatmap.png')

---
## SLIDE 7, 8 — Metrik Karşılaştırmaları

In [ ]:
# Metrik pkl yükleme
def find_metric(base_dir, model, channel, metric='acc.pkl'):
    ch_variants = [channel]
    if 'k3' in channel.lower(): ch_variants = ['rician_k3', 'rician_K3']
    elif 'k10' in channel.lower(): ch_variants = ['rician_k10', 'rician_K10']
    for ch in ch_variants:
        tag = f'{model}_{ch}'
        for sub in ['predictions', '.']:
            p = os.path.join(base_dir, tag, sub, metric) if sub != '.' else os.path.join(base_dir, tag, metric)
            if os.path.exists(p): return pickle.load(open(p, 'rb'))
    return None

models = ['mcldnn', 'petcgdnn']
channels = ['awgn', 'rayleigh', 'rician_k3', 'rician_k10']
mf = {'acc': 'acc.pkl', 'f1': 'f1_macro_scores.pkl', 'mcc': 'mcc_metric_scores.pkl'}
model_titles = {'mcldnn': 'MCLDNN', 'petcgdnn': 'PET-CGDNN'}
ch_labels = {'awgn': 'AWGN', 'rayleigh': 'Rayleigh', 'rician_k3': 'Rician K=3', 'rician_k10': 'Rician K=10'}

scratch = {m: {ch: {mk: find_metric(RESULTS_DIR, m, ch, f) for mk, f in mf.items()} for ch in channels} for m in models}
ft = {m: {ch: {mk: find_metric(FT_DIR, m, ch, f) for mk, f in mf.items()} for ch in channels} for m in models}

print('SCRATCH (results/):')
for m in models:
    for ch in channels:
        s = scratch[m][ch]
        print(f'  {model_titles[m]:>10}/{ch_labels[ch]:<12} acc={"✅" if s["acc"] else "❌"} f1={"✅" if s["f1"] else "❌"} mcc={"✅" if s["mcc"] else "❌"}')
print('FINE-TUNED (fine_tuning_results/):')
for m in models:
    for ch in channels:
        s = ft[m][ch]
        print(f'  {model_titles[m]:>10}/{ch_labels[ch]:<12} acc={"✅" if s["acc"] else "❌"} f1={"✅" if s["f1"] else "❌"} mcc={"✅" if s["mcc"] else "❌"}')

In [ ]:
# SLIDE 7 — Training Strategy Comparison (Rayleigh)
for m in models:
    awgn_acc = scratch[m]['awgn']['acc']
    scratch_acc = scratch[m]['rayleigh']['acc']
    ft_acc = ft[m]['rayleigh']['acc']
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    plotted = False
    if awgn_acc:
        snrs = sorted(awgn_acc.keys())
        ax.plot(snrs, [awgn_acc[s]*100 for s in snrs], 'o-', lw=2.5, ms=6, label='AWGN Baseline', color='#4CAF50')
        plotted = True
    if scratch_acc:
        snrs = sorted(scratch_acc.keys())
        ax.plot(snrs, [scratch_acc[s]*100 for s in snrs], 's--', lw=2.5, ms=6, label='Scratch on Rayleigh', color='#FF9800')
        plotted = True
    if ft_acc:
        snrs = sorted(ft_acc.keys())
        ax.plot(snrs, [ft_acc[s]*100 for s in snrs], '^-', lw=2.5, ms=6, label='Fine-Tuned on Rayleigh', color='#E63946')
        plotted = True
    if not plotted: plt.close(); continue
    ax.set_xlabel('SNR (dB)', fontsize=13); ax.set_ylabel('Classification Accuracy (%)', fontsize=13)
    ax.set_title(f'{model_titles[m]} — Training Strategy Comparison (Rayleigh)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='lower right'); ax.grid(True, linestyle='--', alpha=0.5); ax.set_ylim([0, 105])
    plt.tight_layout()
    fname = f'slide7_training_strategy_{m}.png'
    fig.savefig(os.path.join(SAVE_DIR, fname), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✅ {fname}')

In [ ]:
# SLIDE 8 — F1 & MCC Model Comparison on Rayleigh (Fine-Tuned)
for metric_key, ylabel, ylim, fname in [
    ('f1', 'F1 Score (Macro)', [0, 1.05], 'slide8_model_comp_f1_rayleigh.png'),
    ('mcc', 'MCC', [-0.1, 1.05], 'slide8_model_comp_mcc_rayleigh.png')
]:
    mcl = ft['mcldnn']['rayleigh'][metric_key]
    pet = ft['petcgdnn']['rayleigh'][metric_key]
    if mcl and pet:
        snrs = sorted(mcl.keys())
        fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
        ax.plot(snrs, [mcl[s] for s in snrs], 'o-', lw=2.5, label='MCLDNN', color='#E63946')
        ax.plot(snrs, [pet[s] for s in snrs], 's-', lw=2.5, label='PET-CGDNN', color='#1D3557')
        ax.set_xlabel('SNR (dB)', fontsize=13); ax.set_ylabel(ylabel, fontsize=13)
        ax.set_title(f'Model {ylabel} Comparison on Rayleigh (Fine-Tuned)', fontsize=14, fontweight='bold')
        ax.legend(fontsize=12, loc='lower right'); ax.grid(True, linestyle='--', alpha=0.5); ax.set_ylim(ylim)
        plt.tight_layout()
        fig.savefig(os.path.join(SAVE_DIR, fname), dpi=300, bbox_inches='tight')
        plt.show()
        print(f'✅ {fname}')
    else:
        print(f'❌ {fname} — veri eksik')

In [ ]:
# Performans Özet Tablosu
print('=' * 90)
print(f'{"Model":>10} | {"Channel":>12} | {"Strategy":>12} | {"Avg Acc%":>8} | {"Max Acc%":>8} | {"Avg F1":>7} | {"Avg MCC":>8}')
print('-' * 90)
for m in models:
    for ch in channels:
        for label, src in [('Scratch', scratch), ('Fine-Tuned', ft)]:
            acc = src[m][ch]['acc']
            if acc is None: continue
            f1 = src[m][ch]['f1']; mcc = src[m][ch]['mcc']
            avg_a = np.mean(list(acc.values())) * 100
            max_a = np.max(list(acc.values())) * 100
            avg_f = f'{np.mean(list(f1.values())):.4f}' if f1 else 'N/A'
            avg_m = f'{np.mean(list(mcc.values())):.4f}' if mcc else 'N/A'
            print(f'{model_titles[m]:>10} | {ch_labels[ch]:>12} | {label:>12} | {avg_a:>7.2f}% | {max_a:>7.2f}% | {avg_f:>7} | {avg_m:>8}')
print('=' * 90)

---
## SLIDE 9 — Confusion Matrix

In [ ]:
sys.path.insert(0, PROJECT_DIR)
from amr_all_in_one_core_toolkit import load_data, prepare_data_mcldnn, MCLDNN, calculate_confusion_matrix
import tensorflow as tf
print(f'TF: {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
AWGN_PKL = os.path.join(PROJECT_DIR, 'RML2016.10a_dict.pkl')
RAY_PKL = os.path.join(PROJECT_DIR, 'RML2016.10a_rayleigh.pkl')
WEIGHTS = None
for ext in ['best_weights.keras', 'best_weights.h5', 'best_weights.weights.h5']:
    w = os.path.join(RESULTS_DIR, 'mcldnn_awgn', ext)
    if os.path.exists(w): WEIGHTS = w; break
print(f'AWGN: {"✅" if os.path.exists(AWGN_PKL) else "❌"}')
print(f'Rayleigh: {"✅" if os.path.exists(RAY_PKL) else "❌"}')
print(f'Weights: {"✅ " + str(WEIGHTS) if WEIGHTS else "❌"}')

In [ ]:
if WEIGHTS and os.path.exists(AWGN_PKL) and os.path.exists(RAY_PKL):
    (mods, snrs, lbl_a), _, _, (Xt_a, Yt_a), (_, _, tidx_a) = load_data(AWGN_PKL)
    (_, _, lbl_r), _, _, (Xt_r, Yt_r), (_, _, tidx_r) = load_data(RAY_PKL)
    tf.keras.backend.clear_session()
    model = MCLDNN(weights=None, classes=len(mods))
    model.load_weights(WEIGHTS)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    def get_cm(X, Y, lbl, tidx, snr):
        mask = np.where(np.array([lbl[i][1] for i in tidx]) == snr)
        Xs, Ys = X[mask], Y[mask]
        inp = prepare_data_mcldnn(Xs, Xs, Xs)
        Yh = model.predict(inp['test'], batch_size=400)
        cm, cor, ncor = calculate_confusion_matrix(Ys, Yh, mods)
        return cm, cor/(cor+ncor)

    cm_a, acc_a = get_cm(Xt_a, Yt_a, lbl_a, tidx_a, 18)
    cm_r, acc_r = get_cm(Xt_r, Yt_r, lbl_r, tidx_r, 18)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=150)
    for ax, cm, t in [(ax1, cm_a, f'AWGN ({acc_a*100:.1f}%)'), (ax2, cm_r, f'Rayleigh ({acc_r*100:.1f}%)')]:
        im = ax.imshow(cm*100, cmap='Blues', vmin=0, vmax=100)
        ax.set_title(t, fontsize=13, fontweight='bold')
        ax.set_xticks(np.arange(len(mods))); ax.set_xticklabels(mods, rotation=90, fontsize=9)
        ax.set_yticks(np.arange(len(mods))); ax.set_yticklabels(mods, fontsize=9)
        for i in range(len(mods)):
            for j in range(len(mods)):
                ax.text(j, i, int(np.around(cm[i,j]*100)), ha='center', va='center', fontsize=7,
                        color='darkorange' if i==j else 'black')
        fig.colorbar(im, ax=ax, shrink=0.8)
    fig.suptitle('MCLDNN Confusion Matrix — SNR = +18 dB', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'slide9_confusion_matrix_comparison.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print('✅ slide9_confusion_matrix_comparison.png')
else:
    print('❌ Gerekli dosyalar eksik!')

---
## Özet

In [ ]:
print('\n' + '=' * 60)
print('📁 ÜRETİLEN DOSYALAR')
print('=' * 60)
expected = [
    'slide2_channel_comparison_constellation.png',
    'slide4_sample_iq_waveforms.png',
    'slide5_fading_heatmap.png',
    'slide7_training_strategy_mcldnn.png',
    'slide7_training_strategy_petcgdnn.png',
    'slide8_model_comp_f1_rayleigh.png',
    'slide8_model_comp_mcc_rayleigh.png',
    'slide9_confusion_matrix_comparison.png',
]
for f in expected:
    p = os.path.join(SAVE_DIR, f)
    if os.path.exists(p): print(f'  ✅ {f} ({os.path.getsize(p)/1024:.0f} KB)')
    else: print(f'  ❌ {f}')
print(f'\n📌 Drive > AMR-Project > Sunum_Grafikleri')